# olmOCR Table HTML Fine-Tuning on Colab

This notebook is set up for a GitHub-based feedback loop:

1. Run a smoke test in Colab
2. Save only lightweight run reports into the repo under `reports/runs/`
3. Commit and push those report files to GitHub
4. I can review the run directly from GitHub

Heavy artifacts like checkpoints should stay outside git.

Google Drive is optional. For cheap smoke tests, the notebook can clone the repo from GitHub and generate mock data directly inside Colab.

At the beginning of each run, keep a note about:

- where you are running it
- what the latest run is
- what arrangement you are testing
- what happened in past runs

## 1. Install dependencies

This cell installs only the missing packages we need for the smoke test. It avoids blanket upgrades because those can force a runtime restart and can pull in incompatible versions.

In [1]:
import importlib.util
import subprocess
import sys

required = {
    'trl': 'trl==1.1.0',
    'bitsandbytes': 'bitsandbytes==0.49.2',
    'qwen_vl_utils': 'qwen-vl-utils==0.0.14',
}
missing = [spec for module, spec in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
else:
    print('Required packages already present; skipping pip install.')

Installing missing packages: ['trl==1.1.0', 'bitsandbytes==0.49.2', 'qwen-vl-utils==0.0.14']


## 2. Validate runtime

Fail early if this Colab session is not actually using CUDA-backed PyTorch.

In [2]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('torch cuda:', torch.version.cuda)

if '+cpu' in torch.__version__ or not torch.cuda.is_available():
    raise RuntimeError(
        'This runtime is not GPU-ready for QLoRA. Restart the runtime once after installs, '
        'make sure GPU is enabled, and rerun from the top.'
    )

torch version: 2.10.0+cu128
cuda available: True
torch cuda: 12.8


## 3. Configure workflow mode

Use `USE_DRIVE = False` for the cheapest smoke test path. That mode clones the repo from GitHub and can generate mock data directly in Colab.

In [3]:
USE_DRIVE = False
USE_MOCK_DATA = True
REPO_OWNER = 'yoelrc88'
REPO_NAME = 'ocr-datasheet-qlora'
REPO_REF = 'main'
GITHUB_TOKEN = ''  # optional for private repos
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/finetune-test'
GITHUB_PROJECT_DIR = '/content/ocr-datasheet-qlora'
RUN_NAME = 'run_001'
LOCAL_WORKDIR = '/content/finetune-test'
ARTIFACT_DIR = f'/content/artifacts/{RUN_NAME}'
MAX_SAMPLES = 0
EPOCHS = 1
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 1e-4

## 4. Optional: mount Google Drive

Run this only if `USE_DRIVE = True`. Skip it for GitHub-only smoke tests.

In [4]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Skipping Drive mount; using GitHub/local mode.')

Skipping Drive mount; using GitHub/local mode.


## 5. Prepare the repo and dataset source

If the repo is private, set `GITHUB_TOKEN` above or use Drive mode. This step now downloads a GitHub archive instead of relying on interactive git auth inside Colab.

In [5]:
import io
import os
import platform
import subprocess
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

def download_repo_archive(owner: str, repo: str, ref: str, token: str, out_dir: str) -> None:
    public_url = f'https://codeload.github.com/{owner}/{repo}/zip/refs/heads/{ref}'
    private_url = f'https://api.github.com/repos/{owner}/{repo}/zipball/{ref}'
    headers = {'Accept': 'application/vnd.github+json'}
    if token:
        headers['Authorization'] = f'Bearer {token}'
        url = private_url
    else:
        url = public_url
    request = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            payload = response.read()
    except urllib.error.HTTPError as error:
        raise RuntimeError(
            'GitHub archive download failed. If the repo is private, set GITHUB_TOKEN or switch to USE_DRIVE=True. '
            f'HTTP {error.code}: {error.reason}'
        ) from error
    except Exception as error:
        raise RuntimeError(f'GitHub archive download failed: {error}') from error

    target = Path(out_dir)
    subprocess.run(['rm', '-rf', out_dir], check=False)
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        archive.extractall('/content')
        roots = sorted({name.split('/', 1)[0] for name in archive.namelist() if '/' in name})
    if not roots:
        raise RuntimeError('Downloaded archive did not contain a repository root directory.')
    extracted_root = Path('/content') / roots[0]
    if extracted_root != target:
        subprocess.run(['rm', '-rf', out_dir], check=False)
        extracted_root.rename(target)

if USE_DRIVE:
    PROJECT_DIR = DRIVE_PROJECT_DIR
else:
    PROJECT_DIR = GITHUB_PROJECT_DIR

if not USE_DRIVE:
    download_repo_archive(REPO_OWNER, REPO_NAME, REPO_REF, GITHUB_TOKEN, GITHUB_PROJECT_DIR)
    print(f'Downloaded repo into {GITHUB_PROJECT_DIR}')

if USE_MOCK_DATA:
    subprocess.run(['python', f'{PROJECT_DIR}/scripts/generate_mock_smoke_test_data.py'], check=True)
    DATA_JSONL = f'{PROJECT_DIR}/mock_smoke_test/mock_table_html.jsonl'
else:
    DATA_JSONL = f'{PROJECT_DIR}/examples/table_html_dataset.example.jsonl'

REPORTS_DIR = f'{PROJECT_DIR}/reports/runs'
REPORT_DIR = f'{REPORTS_DIR}/{RUN_NAME}'
Path(REPORT_DIR).mkdir(parents=True, exist_ok=True)
Path(ARTIFACT_DIR).mkdir(parents=True, exist_ok=True)

gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
runtime_info = [
    f'platform={platform.platform()}',
    f'python={platform.python_version()}',
    f'use_drive={USE_DRIVE}',
    f'use_mock_data={USE_MOCK_DATA}',
    f'project_dir={PROJECT_DIR}',
    f'data_jsonl={DATA_JSONL}',
    f'run_name={RUN_NAME}',
    f'report_dir={REPORT_DIR}',
    f'artifact_dir={ARTIFACT_DIR}',
    '',
    'nvidia-smi:',
    gpu_info.stdout if gpu_info.stdout else gpu_info.stderr,
]
Path(f'{REPORT_DIR}/runtime_info.txt').write_text('\n'.join(runtime_info), encoding='utf-8')
print(Path(f'{REPORT_DIR}/runtime_info.txt').read_text(encoding='utf-8'))

Downloaded repo into /content/ocr-datasheet-qlora
platform=Linux-6.6.113+-x86_64-with-glibc2.35
python=3.12.13
use_drive=False
use_mock_data=True
project_dir=/content/ocr-datasheet-qlora
data_jsonl=/content/ocr-datasheet-qlora/mock_smoke_test/mock_table_html.jsonl
run_name=run_001
report_dir=/content/ocr-datasheet-qlora/reports/runs/run_001
artifact_dir=/content/artifacts/run_001

nvidia-smi:
Mon Apr 13 01:24:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|===================

## 6. Copy repo to local runtime for faster work

This keeps training and temp files off Drive while still writing the report folder back into the repo or cloned checkout.

In [6]:
!rm -rf "$LOCAL_WORKDIR"
!cp -R "$PROJECT_DIR" "$LOCAL_WORKDIR"
LOCAL_DATA_JSONL = DATA_JSONL.replace(PROJECT_DIR, LOCAL_WORKDIR)
LOCAL_PROJECT_DIR = LOCAL_WORKDIR
print('LOCAL_PROJECT_DIR =', LOCAL_PROJECT_DIR)
print('LOCAL_DATA_JSONL =', LOCAL_DATA_JSONL)

LOCAL_PROJECT_DIR = /content/finetune-test
LOCAL_DATA_JSONL = /content/finetune-test/mock_smoke_test/mock_table_html.jsonl


## 7. Run the smoke-test training job and capture the log

In [7]:
train_cmd = [
    'python', '-u',
    f'{LOCAL_PROJECT_DIR}/scripts/train_table_html_lora.py',
    '--data-jsonl', LOCAL_DATA_JSONL,
    '--output-dir', ARTIFACT_DIR,
    '--max-samples', str(MAX_SAMPLES),
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--grad-accum', str(GRAD_ACCUM),
    '--learning-rate', str(LEARNING_RATE),
]

print('Running:', ' '.join(train_cmd))
process = subprocess.Popen(
    train_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
log_lines = []
assert process.stdout is not None
for line in process.stdout:
    print(line, end='')
    log_lines.append(line)
process.wait()
Path(f'{REPORT_DIR}/train.log').write_text(''.join(log_lines), encoding='utf-8')
Path(f'{REPORT_DIR}/command.txt').write_text(' '.join(train_cmd), encoding='utf-8')
if process.returncode != 0:
    raise RuntimeError(f'Training failed with return code {process.returncode}')

Running: python -u /content/finetune-test/scripts/train_table_html_lora.py --data-jsonl /content/finetune-test/mock_smoke_test/mock_table_html.jsonl --output-dir /content/artifacts/run_001 --max-samples 0 --epochs 1 --batch-size 1 --grad-accum 8 --learning-rate 0.0001
Loaded 2 total samples
Train=1 Validation=1 Test=0
Loading processor...
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Processor loaded
Loading quantized model...

Fetching 4 files: 100%|██████████| 4/4 [03:59<00:00, 59.91s/it] 

Loading weights: 100%|██████████| 729/729 [01:09<00:00, 10.48it/s, Materializing param=model.visual.patch_embed.proj.weight]
Model loaded
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Initializin

RuntimeError: Training failed with return code 1

## 8. Copy lightweight outputs into the repo report folder

In [ ]:
lightweight_files = [
    'run_config.json',
    'dataset_split_summary.json',
]

copied = []
for name in lightweight_files:
    src = Path(ARTIFACT_DIR) / name
    if src.exists():
        dst = Path(REPORT_DIR) / name
        dst.write_text(src.read_text(encoding='utf-8'), encoding='utf-8')
        copied.append(name)

artifact_manifest = sorted(p.name for p in Path(ARTIFACT_DIR).glob('*'))
Path(f'{REPORT_DIR}/artifact_manifest.txt').write_text('\n'.join(artifact_manifest), encoding='utf-8')
notes_template = '''# Latest

- run name: {run_name}
- date:
- status: running
- where it ran: Colab
- primary goal for this run:

## Arrangement

- dataset:
- target format: HTML
- input type: cropped tables or full pages
- model base: allenai/olmOCR-2-7B-1025
- fine-tuning method: QLoRA
- image sizing:
- epochs: {epochs}
- batch size: {batch_size}
- grad accumulation: {grad_accum}
- LoRA target modules:

## What Changed In This Run

-

## Result Summary

- did training complete:
- any OOM or instability:
- main log takeaway:
- qualitative output takeaway:

## Comparison To Past Runs

- better than:
- worse than:
- unchanged from:

## Next Action

-
'''.format(run_name=RUN_NAME, epochs=EPOCHS, batch_size=BATCH_SIZE, grad_accum=GRAD_ACCUM)
Path(f'{REPORT_DIR}/notes.md').write_text(notes_template, encoding='utf-8')
print('Copied:', copied)
print('Report dir:', REPORT_DIR)
print('Artifact manifest:')
print(Path(f'{REPORT_DIR}/artifact_manifest.txt').read_text(encoding='utf-8'))

## 9. Review the report files that should be committed

In [ ]:
!ls -la "$REPORT_DIR"

## 10. GitHub handoff

After the run:

1. Update `notes.md` in the run folder with the latest status and a short comparison to past runs
2. If you changed the experiment strategy, update `reports/EXPERIMENT_MATRIX.md`
3. Commit only the new folder under `reports/runs/` and any small tracking-doc updates
4. Push it to GitHub
5. Send me the repo link or tell me the run folder name

That gives me a stable place to review the run without SSH or notebook access.

In [ ]:
print('Suggested git commands:')
print(f'git add reports/runs/{RUN_NAME}')
print('git commit -m "add results for ' + RUN_NAME + '"')
print('git push')